# aInvest 量化策略演示

本 Notebook 演示完整的量化策略开发流程：获取数据 → 计算指标 → 回测 → 评估

In [ ]:
import sys
sys.path.insert(0, '..')

from data import Fetcher
from indicators import compute_all
from strategies import MACrossStrategy
from backtest import BacktestEngine, performance_summary
from visualization import plot_result

import pandas as pd

## 1. 获取数据

以平安银行 (000001) 为例，获取 2020 年至今的日线数据。

In [ ]:
df = Fetcher.a_stock('000001', start='2020-01-01')
print(f"数据量: {len(df)} 条")
df.head()

## 2. 计算技术指标

In [ ]:
df = compute_all(df)
df[['close', 'ma5', 'ma10', 'ma20', 'rsi14', 'bb_upper', 'bb_lower']].tail()

## 3. 创建策略并回测

双均线交叉策略：短线上穿长线买入，下穿卖出。

In [ ]:
# 可以调整 short 和 long 参数测试不同周期
strategy = MACrossStrategy(short=5, long=20)
engine = BacktestEngine(initial_capital=100000, commission=0.0003, slippage=0.001)
result = engine.run(df, strategy)

result[['close', 'signal', 'position', 'equity']].tail(10)

## 4. 绩效评估

In [ ]:
metrics = performance_summary(result)
for k, v in metrics.items():
    unit = '%' if any(w in k for w in ['收益率', '波动率', '回撤', '胜率']) else ''
    print(f"{k}: {v}{unit}")

## 5. 可视化

In [ ]:
plot_result(result, title=f"{strategy.name} — 平安银行 (000001)")

---
## 6. 参数优化实验

遍历不同的均线周期组合，找到最优参数。

In [ ]:
results = []
for short in [3, 5, 10]:
    for long in [15, 20, 30, 60]:
        if short >= long:
            continue
        s = MACrossStrategy(short=short, long=long)
        r = BacktestEngine().run(df, s)
        m = performance_summary(r)
        results.append({
            'short': short, 'long': long,
            **m
        })

param_df = pd.DataFrame(results)
param_df.sort_values('夏普比率', ascending=False)